# Cleanup

::: If you are on an AWS-provided event account, the account is cleaned up automatically —
skip this notebook. :::

This notebook tears down every Module-4 resource you deployed:

1. `cdk destroy` the FAST CDK stack (Amplify, Runtime, Memory, Gateway, Cognito, ECR images)
2. Delete the `/FAST-stack/*` SSM parameters
3. Delete the `workshop-tools-gateway-auth` OAuth2 credential provider
4. (Optional) Delete the A2A registry record from notebook 07

It does **not** touch Module 2 or Module 3 resources — those belong to the platform team.


## Step 1: Destroy the FAST CDK Stack

`cdk destroy --force` removes all resources under the `FAST-stack` name. This is destructive
— all conversation history in AgentCore Memory will be lost.


In [ ]:
import os, subprocess, pathlib

FAST_DIR = pathlib.Path("/workshop/fast-agent")
INFRA_DIR = FAST_DIR / "infra-cdk"

if not INFRA_DIR.exists():
    print(f"{INFRA_DIR} not found — FAST already cleaned up?")
else:
    os.chdir(INFRA_DIR)
    subprocess.run(["npx", "cdk", "destroy", "--force"], check=True)
    print("cdk destroy complete")


## Step 2: Delete SSM Parameters

Remove the parameters the agent used to find the gateway, virtual key, and credential provider.


In [ ]:
import os
import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )
ssm = boto3.client("ssm", region_name=REGION)

for name in [
    "/FAST-stack/gateway_url",
    "/FAST-stack/llm_gateway_url",
    "/FAST-stack/llm_gateway_key",
    "/FAST-stack/gateway_credential_provider",
]:
    try:
        ssm.delete_parameter(Name=name)
        print(f"Deleted {name}")
    except ssm.exceptions.ParameterNotFound:
        print(f"{name} already gone")

## Step 3: Delete the OAuth2 Credential Provider

This removes the `workshop-tools-gateway-auth` provider from AgentCore Identity's Token Vault.


In [ ]:
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

try:
    agentcore.delete_oauth2_credential_provider(name="workshop-tools-gateway-auth")
    print("Deleted provider 'workshop-tools-gateway-auth'")
except agentcore.exceptions.ResourceNotFoundException:
    print("Provider already gone")
except Exception as e:
    if "ResourceNotFound" in str(e) or "not found" in str(e).lower():
        print("Provider already gone")
    else:
        raise


## Step 4 (Optional): Delete the Registry Record

Skip this step if you did not run notebook 07.


In [ ]:
try:
    registries = agentcore.list_registries().get("registries", [])
    if not registries:
        print("No registry present — nothing to clean")
    else:
        registry_id = registries[0]["registryId"]
        records = agentcore.list_registry_records(registryId=registry_id).get("registryRecords", [])
        target = next((r for r in records if r["name"] == "workshop_travel_agent"), None)
        if target is None:
            print("No workshop_travel_agent record — nothing to clean")
        else:
            agentcore.delete_registry_record(registryId=registry_id, recordId=target["recordId"])
            print(f"Deleted record {target['recordId']}")
except Exception as e:
    print(f"Could not clean registry record ({e}) — leaving untouched")


## Summary

Module 4 resources are gone. Platform foundations (Module 2 LLM Gateway, Module 3 Registry +
Tools Gateway) are still running — for full workshop cleanup follow the **Workshop Cleanup**
page.
